### datasources

In [148]:
import pandas as pd

In [4]:
dir = './corn_climate_risk_futures_daily_master.csv'
riskfutures = pd.read_csv(dir)

In [22]:
harvestperiodorder = {
            'Off-season': 0,\
            'Planting': 1,\
            'Mid-season': 2,\
            'Harvest': 3,\
            'Peak Harvest': 4
        }
## order according to https://extension.usu.edu/vegetableguide/sweet-corn/harvest-handling

### EDA - Climate Risk and Corn Futures Data

In [11]:
riskfutures.info(verbose=True)
## Helios data doc:"Futures data may have gaps on non-trading days (weekends, holidays)"

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320661 entries, 0 to 320660
Data columns (total 41 columns):
 #   Column                                                    Non-Null Count   Dtype  
---  ------                                                    --------------   -----  
 0   ID                                                        320661 non-null  object 
 1   crop_name                                                 320661 non-null  object 
 2   country_name                                              320661 non-null  object 
 3   country_code                                              320661 non-null  object 
 4   region_name                                               320661 non-null  object 
 5   region_id                                                 320661 non-null  object 
 6   harvest_period                                            320661 non-null  object 
 7   growing_season_year                                       320661 non-null  int64  
 8   date

In [16]:
riskfutures.head(3)

,ID,crop_name,country_name,country_code,region_name,region_id,harvest_period,growing_season_year,date_on,climate_risk_cnt_locations_heat_stress_risk_low,...,futures_zc1_ma_120,futures_zc1_vol_20,futures_zc1_vol_60,futures_zw_zc_spread,futures_zc_zw_ratio,futures_zs_zc_spread,futures_zc_zs_ratio,date_on_year,date_on_month,date_on_year_month
0,8af42722-3f05-4ede-80fc-605e0e2b3b67,Corn: Commodity Tracked,Argentina,AR,Buenos Aires,bffad37a-7c60-432f-984a-8ea83a944311,Harvest,2017,2016-06-15,23,...,375.014583,0.013520,0.015724,48.50,0.898429,727.00,0.371107,2016,6,2016_06
1,54f4ddc5-e7ab-4bfb-ad6a-5649841af563,Corn: Commodity Tracked,Argentina,AR,Buenos Aires,bffad37a-7c60-432f-984a-8ea83a944311,Harvest,2017,2016-06-16,23,...,375.512500,0.013799,0.015792,47.25,0.900000,709.25,0.374835,2016,6,2016_06
2,63a41fce-d371-4295-a58a-dc6491664020,Corn: Commodity Tracked,Argentina,AR,Buenos Aires,bffad37a-7c60-432f-984a-8ea83a944311,Harvest,2017,2016-06-17,23,...,376.122917,0.013442,0.016145,43.50,0.909610,721.75,0.377533,2016,6,2016_06


In [17]:
riskfutures.describe(include='object')

,ID,crop_name,country_name,country_code,region_name,region_id,harvest_period,date_on,date_on_year_month
count,320661,320661,320661,320661,320661,320661,320661,320661,320661
unique,320661,1,11,11,89,89,5,3637,120
top,8af42722-3f05-4ede-80fc-605e0e2b3b67,Corn: Commodity Tracked,Russia,RU,Kabardino-Balkaria,3614b209-e1df-453d-9fec-16b532706c1e,Off-season,2021-01-25,2021_01
freq,1,320661,90875,90875,3635,3635,96408,89,2759


### corn risk and futures price data time range is from 2016-01 to 2025-12 in 11 countries and 89 subregions.  
### EU as a country and its six subregions are missing compared to regional market share data

## EDA 1.1: climate risk data

In [70]:
climaterisk = riskfutures.iloc[:,[2,4]+list(range(8,21))]
climaterisk.head(3)

,country_name,region_name,date_on,climate_risk_cnt_locations_heat_stress_risk_low,climate_risk_cnt_locations_heat_stress_risk_medium,climate_risk_cnt_locations_heat_stress_risk_high,climate_risk_cnt_locations_unseasonably_cold_risk_low,climate_risk_cnt_locations_unseasonably_cold_risk_medium,climate_risk_cnt_locations_unseasonably_cold_risk_high,climate_risk_cnt_locations_excess_precip_risk_low,climate_risk_cnt_locations_excess_precip_risk_medium,climate_risk_cnt_locations_excess_precip_risk_high,climate_risk_cnt_locations_drought_risk_low,climate_risk_cnt_locations_drought_risk_medium,climate_risk_cnt_locations_drought_risk_high
0,Argentina,Buenos Aires,2016-06-15,23,0,0,23,0,0,23,0,0,16,7,0
1,Argentina,Buenos Aires,2016-06-16,23,0,0,23,0,0,23,0,0,14,9,0
2,Argentina,Buenos Aires,2016-06-17,23,0,0,14,1,8,23,0,0,14,8,1


In [81]:
climaterisksum = climaterisk.agg(lambda x:pd.Series([x['country_name'],
                                    x['region_name'],
                                    x['climate_risk_cnt_locations_heat_stress_risk_low':'climate_risk_cnt_locations_heat_stress_risk_high'].sum(),
                                    x['climate_risk_cnt_locations_unseasonably_cold_risk_low':'climate_risk_cnt_locations_unseasonably_cold_risk_high'].sum(),
                                    x['climate_risk_cnt_locations_excess_precip_risk_low':'climate_risk_cnt_locations_excess_precip_risk_high'].sum(),
                                    x['climate_risk_cnt_locations_drought_risk_low':'climate_risk_cnt_locations_drought_risk_high'].sum()
                                   ],
                                   index = ['country_name',
                                            'region_name',
                                            'heat_stress_risk_sum',
                                            'unseasonably_cold_risk_sum',
                                            'excess_precip_risk_sum',
                                            'drought_risk_sum']),
                                          axis=1)

climaterisksumunique = climaterisksum.groupby('region_name')\
              .apply(lambda x:x.astype('str').describe().loc['unique',:],include_groups=False)

In [117]:
climaterisksumunique.astype('str').describe()

unique,country_name,heat_stress_risk_sum,unseasonably_cold_risk_sum,excess_precip_risk_sum,drought_risk_sum
count,89,89,89,89,89
unique,1,2,2,2,2
top,1,1,1,1,1
freq,89,82,82,82,82


### Acccording to data doc, "Locations: Individual points of interest (POIs) within each region"  
### Table above shows that most regions have a fixed number of points of interests across time among all risk types

In [119]:
climaterisksumunique[climaterisksumunique['heat_stress_risk_sum']!=1]
## all regions in Brazil except Bahia experienced a change of POIs 

unique,country_name,heat_stress_risk_sum,unseasonably_cold_risk_sum,excess_precip_risk_sum,drought_risk_sum
region_name,,,,,
Canindeyú,1,2,2,2,2
Goiás,1,2,2,2,2
Mato Grosso,1,2,2,2,2
Mato Grosso do Sul,1,2,2,2,2
Minas Gerais,1,2,2,2,2
Paraná,1,2,2,2,2
Rio Grande do Sul,1,2,2,2,2


In [147]:
climaterisk[climaterisk['date_on']=='2025-12-15'].iloc[:,3:6].sum(axis=0).sum()
## On a given day there are 1053 POIs across the globe. This is a stable measure 
## since POI numbers in each region remain fixed across time

1053